# Step 2 - Does the training loop run?

Before we put the real reward in, we check the training machinery runs at all.
This trains for just a few steps with a FAKE reward (a random number). We are
not looking for the model to get better. We only want it to run start to finish
without crashing.

If this works, the risky part of the project is behind us.

## Turn off torch.compile first

Unsloth tries to compile part of the training step and it crashes on this
model. We run it plain instead. This has to happen before any other import, so
it is the very first cell. It also deletes any half-built compiled files from
the failed run.

In [ ]:
import os, shutil
os.environ["TORCHDYNAMO_DISABLE"] = "1"
shutil.rmtree("unsloth_compiled_cache", ignore_errors=True)
print("torch.compile disabled")

## Setup: get the repo

In [1]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

# Make sure we are inside the repo folder.
if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)

# Always pull the latest code so the runtime is never stale.
subprocess.run(["git", "pull", "--quiet"], check=False)

sys.path.insert(0, os.getcwd())
print("repo ready, at", os.getcwd())

repo ready


## Install the training libraries

`unsloth` loads the model cheaply in 4-bit and gives us fast GRPO. It pulls in
`trl` (which has the GRPO trainer), `transformers`, `peft` and the rest. This
cell takes a few minutes the first time.

In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 35.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 56.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━

## Print the library versions

If the training cell later fails, these versions are the first thing to check,
because the GRPO library changes its function names between versions.

In [3]:
import unsloth          # import first so it can patch the others
import torch, transformers, trl
print("unsloth     ", unsloth.__version__)
print("trl         ", trl.__version__)
print("transformers", transformers.__version__)
print("torch       ", torch.__version__)
print("gpu         ", torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth      2026.7.5
trl          0.24.0
transformers 5.5.0
torch        2.11.0+cu128
gpu          True


## Run the smoke test

This loads Qwen 2.5 3B in 4-bit, adds the small trainable adapters, and runs 5
GRPO steps on a handful of toy questions with a random reward. Loading the model
takes a minute. The 5 steps are slow because the model writes several answers
per question.

Success looks like a short training table and the line:

    training loop finished without crashing

In [6]:
from train.grpo_train import run

trainer = run("configs/baseline.yaml", max_steps=5)

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.5 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 3 | Total steps = 5
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id', 'disable_compile'}) is de

Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Input IDs of shape torch.Size([1, 1064]) with length 1064 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([16, 1064]) with length 1064 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.


TorchRuntimeError: RuntimeError when making fake tensor call
  Explanation: Dynamo failed to run FX node with fake tensors: call_function <built-in method gather of type object at 0x7a8e50ee5180>(*(FakeTensor(..., device='cuda:0', size=(((s47*s87 + 63)//64), 151936)),), **{'dim': -1, 'index': FakeTensor(..., device='cuda:0', size=(((s61*s91 + 63)//64), 1),
               dtype=torch.int64)}): got RuntimeError('Size does not match at dimension 0 expected index torch.Size([((s61*s91 + 63)//64), 1]) to be no larger than self torch.Size([((s47*s87 + 63)//64), 151936]) apart from dimension 1')
  Hint: Your code may result in an error when running in eager. Please double check that your code doesn't contain a similar error when actually running eager/uncompiled. You can do this by removing the `torch.compile` call, or by using `torch.compiler.set_stance("force_eager")`. 

  Developer debug context: 

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compi

In [5]:
!git pull

remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 22 (delta 12), reused 20 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (22/22), 10.01 KiB | 1.67 MiB/s, done.
From https://github.com/ookino/rlvr-argument-mining
   468ddb6..81a9661  main       -> origin/main
Updating 468ddb6..81a9661
Fast-forward
 docs/run_log.md                 |  42 ++++++++-
 notebooks/00_bootstrap.py       |  17 ++--
 notebooks/01_setup_checks.ipynb | 104 +++++++++------------
 notebooks/02_train_smoke.ipynb  | 194 ++++++++++++++++++++++++++++++++++++++++
 reward/ari.py                   |  20 +++--
 train/grpo_train.py             | 105 ++++++++++++++++++++++
 6 files changed, 403 insertions(+), 79 deletions(-)
 create mode 100644 notebooks/02_train_smoke.ipynb
 create mode 100644 train/grpo_train.py


## What to do next

- If it printed **training loop finished without crashing**: step 2 is done.
  Tell Claude and we wire in the real reward.
- If it **errored**: copy the whole error and paste it back. Version mismatches
  in the GRPO library are common and quick to fix once we see the message.